<a id="consulta"></a>

# 04 · Actualizar la evaluación y consultar un partido

Este cuaderno puede ejecutarse solo. Carga los CSV, prepara variables y evalúa los tres métodos
fijos mediante predicciones temporales. No crea otra muestra de test ni repite hiperparámetros.
Muestra una recomendación; tú eliges Global, Pareja, Referencia o los tres para comparar.
Las instrucciones completas están en [docs/guia_de_uso.md](../docs/guia_de_uso.md).

**Datos del estudio (no incluidos):** temporada 2026/27 hasta J9, 88 partidos finalizados; captura del 20 de octubre de 2026.

**Distribución pública.** Se conservan las explicaciones y los resultados agregados del estudio. Los CSV reales y las salidas de partidos individuales no se incluyen. Para ejecutar las celdas, prepara primero tus datos en `data/local/actualizados` siguiendo [docs/guia_de_uso.md](../docs/guia_de_uso.md). Obtener otra captura no garantiza las mismas métricas.


In [ ]:
from pathlib import Path
import os, sys
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")
RAIZ = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
assert (RAIZ / "src/faltas").is_dir(), "Abre el notebook desde la carpeta pública o notebooks/."
sys.path.insert(0,str(RAIZ / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
get_ipython().run_line_magic("matplotlib", "inline")
from faltas.util import leer_json
pd.set_option("display.max_columns", 12)
pd.set_option("display.precision", 4)
plt.rcParams.update({"figure.dpi":110,"axes.spines.top":False,"axes.spines.right":False})
from faltas.datos import datos_por_defecto
CARPETA_ACTUALIZADA = RAIZ / "data/local/actualizados"
DATOS = (datos_por_defecto() if os.environ.get("FALTAS_DATOS")
         else CARPETA_ACTUALIZADA if CARPETA_ACTUALIZADA.exists() else datos_por_defecto())
print("Carpeta de datos elegida:", DATOS.relative_to(RAIZ) if DATOS.is_relative_to(RAIZ) else DATOS)
INFORMES = RAIZ / "reports/investigacion"
from faltas.entrega import cobertura, INFORMES_ACTUALES
from faltas.datos import cargar_datos
INFO_DATOS = cobertura(cargar_datos(DATOS))
print("Captura:", INFO_DATOS["captura_utc"], "| Hasta jornada:", INFO_DATOS["jornada_hasta"])
print("Partidos actuales finalizados:", INFO_DATOS["partidos_actuales"], "| Historial finalizado:", INFO_DATOS["partidos_finalizados"])

**Qué se conserva y qué se vuelve a ajustar.** Los cuadernos 01–03 justifican variables,
parámetros y pesos. Aquí se reutilizan esas recetas: si cambian los datos, se actualizan sus
métricas; si no, se recuperan de caché. Al pedir un encuentro se ajusta el método seleccionado
con los antecedentes disponibles. No se carga un «ganador entrenado» producido obligatoriamente
por el 03. Global significa todos los clubes; pareja, los dos solicitados.


## Comparación actual mediante walk-forward

Se repite el ciclo **ajustar con el pasado disponible → predecir → medir → avanzar** para los
partidos finalizados de la temporada activa. Cada encuentro queda fuera de su propio ajuste;
su resultado puede usarse después para entrenar una predicción posterior. MAE, RMSE y sesgo
resumen los errores de ese recorrido. El método se explica en [notebooks/02_modelos_y_experimentos.ipynb](02_modelos_y_experimentos.ipynb#walk-forward).

Aquí se comparan recetas ya fijadas; no se buscan nuevos hiperparámetros ni se reservan otras
dos jornadas. La recomendación usa menor RMSE
y, en empate, menor MAE. Estas métricas ayudan a elegir; no son un test independiente del ganador.

Al solicitar un encuentro futuro se ajusta el método elegido con todos sus antecedentes
elegibles, incluidos los resultados recientes que ya se utilizaron para medir. No se apartan
esos partidos del ajuste final por haber servido antes como prueba temporal.

In [ ]:
from faltas.modelos import preparar_estudio, predecir_partido, NOMBRES
from faltas.evaluacion import evaluar_temporal, recomendar_modelo, informe_partido
e = preparar_estudio(DATOS)
actual = e.datos.partidos.loc[e.datos.partidos.temporada.eq(e.datos.manifest["temporada_actual"])]
finalizados = actual.loc[actual.es_elegible]
print("Captura:", e.datos.manifest["capturado_utc"])
print("Temporada:", e.datos.manifest["temporada_actual"], "| Partidos finalizados:", len(finalizados))
print("Mayor jornada con resultados:", int(finalizados.jornada.max()), "(puede contener pendientes)")
pred_historicas = evaluar_temporal(e)
recomendado, ranking = recomendar_modelo(pred_historicas)
display(ranking[["modelo","partidos","n","MAE","RMSE","sesgo"]])
print("Modelo recomendado:", NOMBRES[recomendado])
print("Preparado. Quita el # de la última línea de la última celda y ejecútala.")
print("Las preguntas aparecerán en la barra SUPERIOR de VS Code, fuera de la celda.")

## Elige cómo consultar el partido

| Número | Qué obtendrás |
|---|---|
| **1** | Informe de XGBoost global, con 93 entradas. |
| **2** | Informe de XGBoost por pareja, con 29 entradas. |
| **3** | Informe de la referencia equipo/rival. |
| **4** | Los tres: predicciones en columnas contiguas y métricas de cada método. |

La recomendación histórica indica qué opción tuvo menor RMSE; **la elección siempre es manual**.
En la opción 4 se consulta un único partido con la misma fecha de emisión y datos para los tres.
Después de elegir, se pedirán los equipos, la jornada y el inicio si falta en el calendario.

El informe separa tres ámbitos: errores de **cada club**, errores **por equipo en toda la liga**
y errores de **las faltas totales por partido**. Las últimas dos tablas resumen el historial
del método; no indican cuánto se equivocará en el encuentro que vas a consultar.

In [ ]:
from faltas.consulta import consultar_modelos, preparar_informes_consulta

def mostrar_informes_partido(resultados):
    """Presenta uno o tres métodos sin modificar las predicciones ni sus métricas."""
    resumen = preparar_informes_consulta(resultados, pred_historicas)
    varios = len(resultados) > 1
    if varios:
        print("Predicciones del mismo partido: un método por columna.")
        display(resumen["comparacion"].round(2))
    else:
        print("Método utilizado:", NOMBRES[resultados[0]["modelo"]])
        print("Total de faltas previsto:", round(resultados[0]["total"], 2))

    print("Errores históricos de los equipos consultados; n = partidos evaluados de ese club.")
    columnas = ["equipo", "es_local", "prediccion", "n", "MAE", "RMSE", "sesgo",
                "sobreestima_pct", "infraestima_pct"]
    equipos = resumen["equipos"].copy()
    if varios:
        equipos["Método"] = equipos.modelo.map(NOMBRES)
        columnas.insert(0, "Método")
    display(equipos.reindex(columns=columnas))
    print("Captura:", resumen["captura_utc"], "| Emisión común:", resumen["corte_utc"])

    ambitos = [
        ("historico_general", "Resumen de toda la liga: error por equipo",
         "Incluye ambos equipos de cada encuentro: n = 2 × partidos. El MAE mide faltas por equipo."),
        ("historico_total", "Resumen de toda la liga: error del total del partido",
         "Compara predicción local + visitante con faltas totales reales: n = partidos. "
         "Los errores de los equipos pueden compensarse o acumularse."),
    ]
    for clave, titulo, explicacion in ambitos:
        print(titulo)
        print(explicacion)
        tabla = resumen[clave].copy()
        if tabla.empty:
            print("Todavía no hay historial evaluable en este ámbito.")
            continue
        tabla["Método"] = tabla.modelo.map(NOMBRES)
        display(tabla[["Método", "partidos", "n", "MAE", "RMSE", "sesgo",
                       "sobreestima_pct", "infraestima_pct"]])
    return resumen

## Introduce tu partido: responde arriba, no dentro del código

1. En la última línea de la celda siguiente, borra **solo la almohadilla `#`**. Debe quedar
   `informe_actual = consultar_interactivo()`.
2. Pulsa **▶ a la izquierda de esa celda**. Se abrirá un cuadro en la **barra superior de VS Code**,
   fuera del notebook. Ahí se escribe cada respuesta, seguida de **Enter**.
3. El cuadro preguntará, en orden: **método (1–4) → local → visitante → jornada → inicio**.
   Escribe solo el número de jornada. Para el inicio, Enter usa el calendario de los CSV.

**Nombres de los equipos:** abre `equipos.csv` dentro de la carpeta indicada en «Carpeta de datos
elegida». Normalmente será `data/local/actualizados/equipos.csv`. Si seleccionaste otra carpeta con
`FALTAS_DATOS`, consulta su `equipos.csv`. Copia el texto exacto de la columna **`nombre`** para cada equipo
(también se admite su `id_equipo`). No cambies las líneas `input(...)` del código.

Mientras espera tus respuestas, la celda puede mostrar `[*]`: mira la barra superior.
Tras la última respuesta, el informe aparecerá **debajo de la celda**. Para otra consulta,
ejecuta solo esta última celda; si cambias CSV, reinicia el kernel y ejecuta el 04 completo.

**Cómo leer las métricas:** MAE es error absoluto medio; RMSE penaliza más los fallos grandes.
Sesgo positivo significa sobreestimación media y negativo, infraestimación. Son errores históricos,
no una banda de certeza del próximo partido. No se corrige la predicción con el sesgo ni se escribe en Excel.

In [ ]:
from faltas.opciones_consulta import MODELOS_CONSULTA, resolver_modelos_consulta

def consultar_interactivo():
    """Pide un único partido real y permite consultar uno o los tres métodos."""
    print("Temporada:", e.datos.manifest["temporada_actual"])
    opcion_recomendada = MODELOS_CONSULTA.index(recomendado) + 1
    print("Recomendación histórica:", NOMBRES[recomendado], f"(opción {opcion_recomendada})")
    print("1 · Global (93) | 2 · Pareja (29) | 3 · Referencia | 4 · Comparar los tres")
    print("Responde en la barra SUPERIOR de VS Code; confirma cada respuesta con Enter.")
    print("Copia los equipos de la columna 'nombre' de:", DATOS / "equipos.csv")
    while True:
        opcion = input("Método [1=Global, 2=Pareja, 3=Referencia, 4=Los tres]: ").strip()
        try:
            modelos = resolver_modelos_consulta(opcion)
            break
        except ValueError as error:
            print(error)
    print("Métodos elegidos:", ", ".join(NOMBRES[m] for m in modelos))
    local = input("Equipo local (copia su nombre de equipos.csv): ").strip()
    visitante = input("Equipo visitante (copia su nombre de equipos.csv): ").strip()
    jornada_texto = input("Jornada (solo número): ").strip()
    if not local or not visitante or not jornada_texto.isdigit() or int(jornada_texto) < 1:
        raise ValueError("Escribe ambos equipos y una jornada positiva.")
    inicio = input("Inicio con zona horaria si falta en calendario (Enter para usar CSV): ").strip() or None
    resultados = consultar_modelos(e, local, visitante, opcion=opcion, jornada=int(jornada_texto),
        fecha_inicio_utc=inicio)
    return mostrar_informes_partido(resultados)

# informe_actual = consultar_interactivo()